In [2]:
#run refresh_trade_views to init arbitrage tables
#from db.auto_repo_sqlite import snapshot_many, TableSpec, upsert_many
#from domain.market_rows import MarketGoodRow, MarketTransactionRow
#import sqlite3


from __future__ import annotations
import asyncio
from typing import Optional, List
from openapi_client.api.fleet_api import FleetApi
from openapi_client.api.agents_api import AgentsApi
from openapi_client.api.systems_api import SystemsApi

from runtime_support import (
    setup_client_from_env,
    api_navigate_ship,
    api_get_ship_nav,
    api_patch_nav_flight_mode,
    api_purchase_cargo,
    api_sell_cargo,
    build_fleet_object,
    api_create_ship_waypoint_scan
    )

from core_helpers import init_world_state, load_initial_fleet_state

from core_helpers import build_nodes_from_traits_dict
from refuel_routing import plan_route_and_refuel_with_reserve

from db.refresh_trade_views import refresh_trade_views
from services.arbitrage_repo import top_arbitrage

async def trade_loop(
    fleet_api,
    ship,
    route_buy_point,
    route_sell_point,
    arbi_buy_wp,
    arbi_sell_wp,
    arbi_trade_symbol,
    units=20
):
    #while True:  # repeat forever, or replace with a counter/condition

        # Navigate to buy waypoint(s)
        for rt in route_buy_point:
            await api_navigate_ship(fleet_api, ship, rt)

        # Dock and buy
        await api_purchase_cargo(fleet_api, ship, arbi_buy_wp, arbi_trade_symbol, units)
        await api_purchase_cargo(fleet_api, ship, arbi_buy_wp, arbi_trade_symbol, units)

        # Navigate to sell waypoint(s)
        for rt in route_sell_point:
            await api_navigate_ship(fleet_api, ship, rt)

        # Dock and sell
        await api_sell_cargo(fleet_api, ship, arbi_sell_wp, arbi_trade_symbol, units)
        await api_sell_cargo(fleet_api, ship, arbi_sell_wp, arbi_trade_symbol, units)

def check_cost(route_list: list):
    if not route_list:
        return(0)
    else:
        total = sum(int(rl[-1]) for rl in route_list)
        return(int(total))

def a_b_dist(coord1: str, coord2: str):

    import math
    x1 = world_state.waypoints.by_symbol[coord1].x
    y1 = world_state.waypoints.by_symbol[coord1].y
    x2 = world_state.waypoints.by_symbol[coord2].x
    y2 = world_state.waypoints.by_symbol[coord2].y

    x = x1-x2
    y = y1-y2

    return(math.hypot(x,y))

#---------------------------------------------------------

with setup_client_from_env() as client:
    fleet_api = FleetApi(client)
    agents_api = AgentsApi(client)
    systems_api = SystemsApi(client)


world_state = await init_world_state(fleet_api, agents_api, systems_api)

# --------- define ship roles -----------
fleet_state = await load_initial_fleet_state(fleet_api)

# Extract symbol for a given role
def get_symbol_by_role(ships_dict, role):
    for ship in ships_dict.values():
        if ship.role == role:
            return ship.symbol
    return None

command_ship = get_symbol_by_role(fleet_state.specs, "COMMAND")
satellite = get_symbol_by_role(fleet_state.specs, "SATELLITE")


In [38]:

# Create Survey
api_response =  fleet_api.create_survey(command_ship)

from openapi_client import Configuration, ApiClient
cfg = Configuration()
api_client = ApiClient(cfg)

# print(api_client.sanitize_for_serialization(api_response))
surveys = api_client.sanitize_for_serialization(api_response)

# list all surveys
surveys['data']['surveys']


[{'signature': 'X1-XG6-BD5A-BC0480',
  'symbol': 'X1-XG6-BD5A',
  'deposits': [{'symbol': 'ALUMINUM_ORE'},
   {'symbol': 'COPPER_ORE'},
   {'symbol': 'IRON_ORE'},
   {'symbol': 'COPPER_ORE'}],
  'expiration': '2025-09-20T01:01:42.793Z',
  'size': 'MODERATE'},
 {'signature': 'X1-XG6-BD5A-F960FD',
  'symbol': 'X1-XG6-BD5A',
  'deposits': [{'symbol': 'QUARTZ_SAND'},
   {'symbol': 'SILICON_CRYSTALS'},
   {'symbol': 'QUARTZ_SAND'},
   {'symbol': 'COPPER_ORE'},
   {'symbol': 'ICE_WATER'},
   {'symbol': 'ICE_WATER'},
   {'symbol': 'COPPER_ORE'}],
  'expiration': '2025-09-20T00:25:02.793Z',
  'size': 'SMALL'}]

In [40]:
payload = surveys['data']['surveys'][0]

resp = fleet_api.extract_resources_with_survey(ship_symbol=command_ship, survey=payload)

clean_resp = api_client.sanitize_for_serialization(resp)
clean_resp['data']

{'extraction': {'shipSymbol': 'DDDD-1',
  'yield': {'symbol': 'IRON_ORE', 'units': 5}},
 'cooldown': {'shipSymbol': 'DDDD-1',
  'totalSeconds': 80,
  'remainingSeconds': 79,
  'expiration': '2025-09-20T00:13:14.554000+00:00'},
 'cargo': {'capacity': 40,
  'units': 22,
  'inventory': [{'symbol': 'ALUMINUM_ORE',
    'name': 'Aluminum Ore',
    'description': 'A valuable ore used in the production of aluminum and other alloys. Aluminum ore is an essential component in the construction of ship hulls and other structural components.',
    'units': 5},
   {'symbol': 'IRON_ORE',
    'name': 'Iron Ore',
    'description': 'A common and valuable ore used in the production of steel and other alloys.',
    'units': 5},
   {'symbol': 'QUARTZ_SAND',
    'name': 'Quartz Sand',
    'description': 'High-purity quartz sand used in the production of glass and ceramics.',
    'units': 5},
   {'symbol': 'COPPER_ORE',
    'name': 'Copper Ore',
    'description': 'A valuable ore used in the production of co

In [41]:
await api_navigate_ship(fleet_api, command_ship, 'X1-XG6-H57')

starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Not at destination, continuing
Refuelling now...
Not in orbit, going into orbit now...
Prep complete
DDDD-1  has taken off and is in transit
[BOOT] Adapted 2 ships into fleet_object
Seconds until arrival: 32.384986
Arrived and ready


NavigateShip200ResponseData(nav=ShipNav(system_symbol='X1-XG6', waypoint_symbol='X1-XG6-H57', route=ShipNavRoute(destination=ShipNavRouteWaypoint(symbol='X1-XG6-H57', type=<WaypointType.MOON: 'MOON'>, system_symbol='X1-XG6', x=17, y=-42), origin=ShipNavRouteWaypoint(symbol='X1-XG6-BD5A', type=<WaypointType.ENGINEERED_ASTEROID: 'ENGINEERED_ASTEROID'>, system_symbol='X1-XG6', x=-5, y=-28), departure_time=datetime.datetime(2025, 9, 20, 0, 12, 35, 369000, tzinfo=TzInfo(UTC)), arrival=datetime.datetime(2025, 9, 20, 0, 13, 8, 369000, tzinfo=TzInfo(UTC))), status=<ShipNavStatus.IN_TRANSIT: 'IN_TRANSIT'>, flight_mode=<ShipNavFlightMode.CRUISE: 'CRUISE'>), fuel=ShipFuel(current=374, capacity=400, consumed=ShipFuelConsumed(amount=26, timestamp=datetime.datetime(2025, 9, 20, 0, 12, 35, 372000, tzinfo=TzInfo(UTC)))), events=[])